# Market Data Module V2 (Standalone)

Produces a sorted list of valid price tiers per `(product_id, region)` by combining four market data sources. Also provides margin tiers, market signals, and legacy percentile output. Fully independent — no V1 dependency.

## Sources
1. **Ben Soliman (savvy)** — reference distributor prices from manual savvy mapping (all regions)
2. **Ben Soliman (in-house)** — reference distributor prices from algorithmic in-house mapping (all regions)
3. **Marketplace** — live shelf prices from MaxAB marketplace sellers (regional, with fallback)
4. **Scraped** — competitor prices from Cartona, Tawfeer, Talabia (regional, with fallback; Speed = Alex only)

## Pipeline
Collect prices → regional fallback → floor filter (>= 0.9 x WAC) → brand fallback → commercial group union → step subdivision → output

## Usage
```python
%run market_data_module_2.ipynb

# V2 output: sorted price tier lists for pricing decisions
df_market_v2 = get_market_data_v2()
df_market_cohorts = expand_to_cohorts(df_market_v2)

# Legacy output: percentile columns derived from V2 price_tiers
df_market_legacy = get_market_data_legacy()

# Margin tiers: historical realized margins per warehouse x product
df_margin_tiers = get_margin_tiers()

# Market signals: 60-day technical indicators
df_signals = get_market_signals()
```

In [ ]:
# =============================================================================
# SETUP (standalone — no V1 dependency)
# =============================================================================
import os as _os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# Add parent directory for shared modules
sys.path.insert(0, _os.path.abspath(_os.path.join(_os.getcwd(), '..')))
sys.path.insert(0, _os.path.abspath('.'))

import setup_environment_2
setup_environment_2.initialize_env()
from db import query_snowflake

TIMEZONE = 'America/Los_Angeles'
CAIRO_TZ = pytz.timezone('Africa/Cairo')

# Load queries_module for commercial functions + margin boundary fallbacks
_qm_path = 'queries_module.ipynb' if _os.path.exists('queries_module.ipynb') else 'modules/queries_module.ipynb'
if _os.path.exists(_qm_path):
    %run $_qm_path

ALL_REGIONS = ['Cairo', 'Giza', 'Alexandria', 'Delta East', 'Delta West', 'Upper Egypt']

REGIONAL_FALLBACK = {
    'Cairo':       ['Giza'],
    'Giza':        ['Cairo'],
    'Delta East':  ['Delta West'],
    'Delta West':  ['Delta East'],
    'Alexandria':  ['Delta East', 'Delta West', 'Cairo', 'Giza'],
    'Upper Egypt': ['Cairo', 'Giza'],
}

REGION_COHORT_MAP = {
    'Cairo': [700], 'Giza': [701], 'Alexandria': [702],
    'Delta West': [703], 'Delta East': [704],
    'Upper Egypt': [1123, 1124, 1125, 1126],
}

DEFAULT_TARGET_MARGIN = 0.10
MAX_MARGIN_GAP_PCT = 0.30

print(f'\nMarket Data Module V2 ready (standalone)')
print('Functions: get_market_data_v2(), get_market_data_legacy(), get_margin_tiers(), get_market_signals(), expand_to_cohorts()')

In [ ]:
# =============================================================================
# SOURCE 1: BEN SOLIMAN
# Two-track logic: main (diff < 30%) and lower (diff > 30% with PU correction)
# Not regional - one price per product, applied to all regions
# =============================================================================
V2_BEN_QUERY = f'''
WITH lower_raw AS (
    SELECT
        sm.maxab_product_id AS product_id,
        sm.injection_date,
        sm.bs_price,
        f.wac1,
        f.wac_p,
        sm.maxab_basic_unit_count,
        cpc.price AS cu_price,
        pup.child_quantity,
        ABS(sm.bs_price - (f.wac_p * sm.maxab_basic_unit_count)) / NULLIF(f.wac_p * sm.maxab_basic_unit_count, 0) AS diff,
        ROUND(cpc.price / sm.bs_price) AS p1
    FROM materialized_views.savvy_mapping sm
    JOIN finance.all_cogs f
        ON f.product_id = sm.maxab_product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    JOIN packing_unit_products pu
        ON pu.product_id = sm.maxab_product_id AND pu.is_basic_unit = 1
    JOIN cohort_product_packing_units cpc
        ON cpc.product_packing_unit_id = pu.id AND cpc.cohort_id = 700
    JOIN packing_unit_products pup
        ON pup.product_id = sm.maxab_product_id AND pup.is_basic_unit = 1
    WHERE sm.bs_price IS NOT NULL
        AND sm.injection_date::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 5
),

-- Main track: BS price close to our cost (diff < 30%%)
main_track AS (
    SELECT
        product_id,
        AVG(bs_price / maxab_basic_unit_count) AS ben_soliman_price,
        MAX(injection_date) AS injection_date
    FROM (
        SELECT *,
            bs_price / maxab_basic_unit_count AS unit_price,
            (bs_price / maxab_basic_unit_count - wac_p) / NULLIF(wac_p, 0) AS margin_diff,
            ROW_NUMBER() OVER (PARTITION BY product_id, bs_price ORDER BY diff) AS rnk
        FROM lower_raw
        WHERE diff < 0.3
        QUALIFY MAX(injection_date::DATE) OVER (PARTITION BY product_id) = injection_date::DATE
    )
    WHERE rnk = 1
        AND margin_diff BETWEEN -0.5 AND 0.5
    GROUP BY product_id
    HAVING ben_soliman_price BETWEEN MIN(wac_p) * 0.8 AND MIN(wac_p) * 1.3
),

-- Lower track: BS price far from cost (diff > 30%%), needs PU correction + WAC sanity check
lower_track AS (
    SELECT
        product_id,
        AVG(adjusted_price) AS ben_soliman_price,
        MAX(injection_date) AS injection_date
    FROM (
        SELECT
            product_id,
            injection_date,
            wac_p,
            bs_price * multiplier AS adjusted_price
        FROM (
            SELECT *,
                CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END AS scheck,
                CASE
                    WHEN (ROUND(p1 / NULLIF(CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END, 0))
                          * (CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END)) = 0
                    THEN ROUND(p1 / 2) * 2
                    ELSE ROUND(p1 / NULLIF(CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END, 0))
                         * (CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END)
                END AS multiplier
            FROM lower_raw
            WHERE diff > 0.3 AND p1 > 1
            QUALIFY MAX(injection_date::DATE) OVER (PARTITION BY product_id) = injection_date::DATE
        )
    )
    WHERE adjusted_price BETWEEN wac_p * 0.9 AND wac_p * 1.2
    GROUP BY product_id
),

-- Combine: main track preferred (prio 1), lower track fallback (prio 2)
combined AS (
    SELECT product_id, ben_soliman_price, injection_date, 1 AS prio FROM main_track
    UNION ALL
    SELECT product_id, ben_soliman_price, injection_date, 2 AS prio FROM lower_track
),

best AS (
    SELECT *
    FROM combined
    QUALIFY MAX(injection_date) OVER (PARTITION BY product_id) = injection_date
        AND prio = MIN(prio) OVER (PARTITION BY product_id)
)

SELECT product_id, AVG(ben_soliman_price) AS price
FROM best
GROUP BY product_id
'''

print('Source 1: Ben Soliman query defined')

In [ ]:
# =============================================================================
# SOURCE 2: MARKETPLACE
# Live shelf prices from MaxAB marketplace sellers, per region
# Cleaning: WAC4 +/-40% filter, IQR on ticket size and max_per_order
# Output: per-basic-unit price per (product, region)
# =============================================================================
V2_MARKETPLACE_QUERY = f'''
WITH seller_region AS (
    SELECT
        seller_retailer.retailer_id,
        CASE WHEN regions.name_en = 'Greater Cairo' THEN cities.name_en
             ELSE regions.name_en END AS region,
        seller_id
    FROM materialized_views.sellers_retailers_mapping seller_retailer
    JOIN retailers ON retailers.id = seller_retailer.retailer_id
    JOIN materialized_views.retailer_polygon ON materialized_views.retailer_polygon.retailer_id = retailers.id
    JOIN districts ON districts.id = materialized_views.retailer_polygon.district_id
    JOIN cities ON cities.id = districts.city_id
    JOIN states ON states.id = cities.state_id
    JOIN regions ON regions.id = states.region_id
    JOIN egypt_marketplace.sellers ON sellers.id = seller_retailer.seller_id AND sellers.status = 'ACTIVE'
    where  seller_id <> 74
),

recent_price AS (
    SELECT wp.product_id, wp.packing_unit_id AS product_pu, wp.price,
           wp.max_per_order, warehouses.seller_id, warehouses.min_ticket_size
    FROM egypt_marketplace.warehouse_products wp
    LEFT JOIN egypt_marketplace.warehouses ON warehouses.id = wp.warehouse_id and deleted_at is null 
    WHERE wp.available > 0 AND wp.total_stock > 0 AND activation = 'true'
),

mp_raw AS (
    SELECT DISTINCT sr.region, rp.product_id, rp.price,
           rp.max_per_order, rp.seller_id, rp.min_ticket_size
    FROM recent_price rp
    JOIN seller_region sr ON sr.seller_id = rp.seller_id
),

pu_wacs AS (
    SELECT pup.product_id, pup.packing_unit_id AS pu_id,
           pup.basic_unit_count AS buc, f.wac4,
           pup.basic_unit_count * f.wac4 AS pu_wac4
    FROM packing_unit_products pup
    LEFT JOIN finance.all_cogs f ON f.product_id = pup.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
),

mp_filtered AS (
    SELECT m.region, m.product_id, m.price / pw.buc AS unit_price,
           m.max_per_order, m.min_ticket_size
    FROM mp_raw m
    JOIN pu_wacs pw ON pw.product_id = m.product_id
    WHERE pw.pu_wac4 > 0
        AND ROUND((m.price - pw.pu_wac4) / pw.pu_wac4 * 100, 2) BETWEEN -40 AND 40
),

ticket_iqr AS (
    SELECT *,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY min_ticket_size)
            OVER (PARTITION BY region, product_id) AS ts_q1,
        PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY min_ticket_size)
            OVER (PARTITION BY region, product_id) AS ts_q3
    FROM mp_filtered
),
ticket_cleaned AS (
    SELECT * FROM ticket_iqr
    WHERE min_ticket_size >= ts_q1 - 1.5 * (ts_q3 - ts_q1)
      AND min_ticket_size <= ts_q3 + 3.0 * (ts_q3 - ts_q1)
),

order_iqr AS (
    SELECT *,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY max_per_order)
            OVER (PARTITION BY region, product_id) AS mo_q1,
        PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY max_per_order)
            OVER (PARTITION BY region, product_id) AS mo_q3
    FROM ticket_cleaned
),
cleaned AS (
    SELECT region, product_id, unit_price AS price
    FROM order_iqr
    WHERE max_per_order >= mo_q1 - 1.5 * (mo_q3 - mo_q1)
)

SELECT region, product_id, price, 'marketplace' AS source
FROM cleaned
'''

print('Source 2: Marketplace query defined')

In [ ]:
# =============================================================================
# SOURCE 3: SCRAPED
# Competitor prices: Cartona, Tawfeer, Talabia (Speed = Alexandria only)
# Cartona: keep where min_ticket_size < 8000 AND quantity_per_unit > 5
# Match to closest PU by WAC (within 30% diff)
# =============================================================================
V2_SCRAPED_QUERY = f'''
WITH latest_per_sku AS (
    SELECT product_name_ar, MAX(created_at::DATE) AS max_date
    FROM materialized_views.raw_scraped_data
    WHERE created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 4
    GROUP BY product_name_ar
),

main_prices AS (
    SELECT DISTINCT r.*
    FROM materialized_views.raw_scraped_data r
    JOIN latest_per_sku l ON r.product_name_ar = l.product_name_ar AND r.created_at::DATE = l.max_date
    WHERE (
        (r.source_app = 'Speed' AND COALESCE(r.region, r.area) IN ('Alex', 'Alexandria'))
        OR r.source_app IN ('Cartona', 'Tawfeer', 'Talabia')
    )
),

cleaned_data AS (
    SELECT cm.query_id AS product_id,
           CASE COALESCE(mp.region, mp.area)
               WHEN 'Giza' THEN 'Cairo'
               WHEN 'Alex' THEN 'Alexandria'
               ELSE COALESCE(mp.region, mp.area)
           END AS region,
           mp.price AS comp_price,
           mp.source_app
    FROM main_prices mp
    JOIN materialized_views.competitors_mapping_fixed cm ON cm.match = mp.product_name_ar
),

matched_prices AS (
    SELECT cd.product_id, cd.region,
           cd.comp_price / pup.basic_unit_count AS price,
           cd.source_app AS source
    FROM cleaned_data cd
    JOIN finance.all_cogs f ON f.product_id = cd.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    JOIN packing_unit_products pup ON pup.product_id = cd.product_id
    WHERE ABS(cd.comp_price - f.wac1 * pup.basic_unit_count) / NULLIF(f.wac1 * pup.basic_unit_count, 0) < 0.3
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cd.product_id, COALESCE(cd.source_app, 'unknown'), cd.region
        ORDER BY ABS(cd.comp_price - f.wac1 * pup.basic_unit_count) / NULLIF(f.wac1 * pup.basic_unit_count, 0)
    ) = 1
)

SELECT region, product_id, price, source
FROM matched_prices
'''

print('Source 3: Scraped query defined')

In [ ]:
# =============================================================================
# SUPPORTING QUERIES
# =============================================================================

V2_WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN from_date AND to_date
'''

V2_TARGET_MARGINS_QUERY = f'''
WITH brand_cat_target AS (
    SELECT DISTINCT
        b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_bm
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    QUALIFY CASE
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
    END = DATE_TRUNC('month', date)
),
cat_target AS (
    SELECT cat,
           SUM(target_bm * (target_nmv / NULLIF(cat_total, 0))) AS cat_target_margin
    FROM (
        SELECT *, SUM(target_nmv) OVER (PARTITION BY cat) AS cat_total
        FROM (
            SELECT cat, brand, AVG(target_bm) AS target_bm, SUM(target_nmv) AS target_nmv
            FROM (
                SELECT DISTINCT date, cat, brand, margin AS target_bm, nmv AS target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE
                    WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
                    THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
                    ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
                END = DATE_TRUNC('month', date)
            ) GROUP BY ALL
        )
    ) GROUP BY ALL
)
SELECT DISTINCT bct.cat, bct.brand, bct.target_bm, ct.cat_target_margin
FROM brand_cat_target bct
LEFT JOIN cat_target ct ON ct.cat = bct.cat
'''

V2_PRODUCT_QUERY = '''
SELECT p.id AS product_id, b.name_ar AS brand, c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
WHERE p.activation = 'true'
'''

V2_GROUPS_QUERY = 'SELECT * FROM materialized_views.sku_commercial_groups_pp'

V2_ATH_MARGIN_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
daily_margin_data AS (
    SELECT
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.created_at::DATE AS sale_date,
        SUM(pso.total_price) AS daily_nmv,
        SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count) AS daily_cogs,
        CASE
            WHEN SUM(pso.total_price) > 0
            THEN (SUM(pso.total_price) - SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count)) / SUM(pso.total_price)
            ELSE 0
        END AS daily_margin
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    JOIN finance.all_cogs f ON f.product_id = pso.product_id
        AND f.from_date::DATE <= so.created_at::DATE
        AND f.to_date::DATE > so.created_at::DATE
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 240
        AND so.created_at::DATE < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY COALESCE(pw.parent_id, pso.warehouse_id), pso.product_id, so.created_at::DATE
),
moving_stats AS (
    SELECT
        curr.warehouse_id, curr.product_id, curr.sale_date,
        curr.daily_margin, curr.daily_nmv,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY hist.daily_margin) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY hist.daily_margin) AS q3
    FROM daily_margin_data curr
    JOIN daily_margin_data hist
      ON curr.warehouse_id = hist.warehouse_id
      AND curr.product_id = hist.product_id
      AND hist.sale_date BETWEEN DATEADD(day, -30, curr.sale_date) AND curr.sale_date
    GROUP BY ALL
),
iqr_filtered AS (
    SELECT product_id, warehouse_id, MAX(daily_margin) AS max_margin
    FROM (
        SELECT *,
            (q3 - q1) AS iqr,
            (q1 - 1.5 * (q3 - q1)) AS lower_bound,
            (q3 + 1.5 * (q3 - q1)) AS upper_bound
        FROM moving_stats
        WHERE daily_margin BETWEEN (q1 - 1.5 * (q3 - q1)) AND (q3 + 1.5 * (q3 - q1))
        GROUP BY ALL
    )
    GROUP BY ALL
)
SELECT
    product_id,
    CASE WHEN MAX(max_margin) < 0 THEN 0.07 ELSE MAX(max_margin) END AS ath_margin
FROM iqr_filtered
GROUP BY product_id
'''

V2_BEN_INHOUSE_QUERY = '''
SELECT DISTINCT bim.product_id, bs_matched_price / maxab_buc AS price
FROM MATERIALIZED_VIEWS.bensoliman_inhouse_mapping bim
JOIN finance.all_cogs f
  ON f.product_id = bim.product_id
  AND CURRENT_TIMESTAMP BETWEEN f.from_date AND f.to_date
WHERE bs_matched_price / maxab_buc > 0.9 * wac_p
'''

print('Supporting queries defined')

In [ ]:
# =============================================================================
# MARGIN TIERS + MARKET SIGNALS (moved from V1)
# =============================================================================

MARGIN_BOUNDARIES_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
all_daily_margins AS (
    SELECT
        so.created_at::DATE AS order_date,
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        SUM(pso.total_price) AS nmv,
        SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count) AS cogs,
        DATEDIFF('day', so.created_at::DATE,
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        ) AS days_ago,
        EXTRACT(QUARTER FROM so.created_at::DATE) AS order_quarter
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so
        ON so.id = pso.sales_order_id
    JOIN COHORT_PRICING_CHANGES cpc
        ON cpc.id = pso.COHORT_PRICING_CHANGE_id
    JOIN finance.all_cogs f
        ON f.product_id = pso.product_id
        AND f.from_date::DATE <= so.created_at::DATE
        AND f.to_date::DATE  >  so.created_at::DATE
    WHERE so.created_at::DATE BETWEEN
            DATEADD('year', -1,
                CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
            AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
        AND cpc.cohort_id IN (700, 701, 702, 703, 704, 1123, 1124, 1125, 1126)
    GROUP BY ALL
    HAVING nmv > 0
),

daily_with_margin AS (
    SELECT
        *,
        (nmv - cogs) / NULLIF(nmv, 0) AS daily_margin,
        nmv - cogs AS gross_profit
    FROM all_daily_margins
),

iqr_stats AS (
    SELECT
        product_id,
        warehouse_id,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY daily_margin) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY daily_margin) AS q3
    FROM daily_with_margin
    GROUP BY product_id, warehouse_id
),

iqr_cleaned AS (
    SELECT dm.*
    FROM daily_with_margin dm
    JOIN iqr_stats iq
        ON dm.product_id = iq.product_id
        AND dm.warehouse_id = iq.warehouse_id
    WHERE dm.daily_margin
        BETWEEN iq.q1 - 1.5 * (iq.q3 - iq.q1)
            AND iq.q3 + 1.5 * (iq.q3 - iq.q1)
),

same_quarter_data AS (
    SELECT *
    FROM iqr_cleaned
    WHERE order_quarter = EXTRACT(QUARTER FROM
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
),

boundary_with_weights AS (
    SELECT *, EXP(-0.023 * days_ago) AS time_weight
    FROM same_quarter_data
),

boundary_ordered AS (
    SELECT
        product_id, warehouse_id, daily_margin, time_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY daily_margin
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS total_weight,
        COUNT(*) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS data_points
    FROM boundary_with_weights
),

quarter_percentiles AS (
    SELECT
        product_id, warehouse_id,
        MIN(CASE WHEN cum_weight >= total_weight * 0.10 THEN daily_margin END) AS MIN_BOUNDARY,
        MIN(CASE WHEN cum_weight >= total_weight * 0.50 THEN daily_margin END) AS MEDIAN_BM,
        MIN(CASE WHEN cum_weight >= total_weight * 0.90 THEN daily_margin END) AS MAX_BOUNDARY
    FROM boundary_ordered
    WHERE data_points >= 5
    GROUP BY product_id, warehouse_id
),

full_year_with_weights AS (
    SELECT *, EXP(-0.023 * days_ago) AS time_weight
    FROM iqr_cleaned
),

full_year_ordered AS (
    SELECT
        product_id, warehouse_id, daily_margin, time_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY daily_margin
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS total_weight,
        COUNT(*) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS data_points
    FROM full_year_with_weights
),

full_year_percentiles AS (
    SELECT
        product_id, warehouse_id,
        MIN(CASE WHEN cum_weight >= total_weight * 0.10 THEN daily_margin END) AS MIN_BOUNDARY,
        MIN(CASE WHEN cum_weight >= total_weight * 0.50 THEN daily_margin END) AS MEDIAN_BM,
        MIN(CASE WHEN cum_weight >= total_weight * 0.98 THEN daily_margin END) AS MAX_BOUNDARY
    FROM full_year_ordered
    WHERE data_points >= 5
    GROUP BY product_id, warehouse_id
),

weighted_percentiles AS (
    SELECT
        COALESCE(qp.product_id,   fy.product_id)   AS product_id,
        COALESCE(qp.warehouse_id, fy.warehouse_id) AS warehouse_id,
        COALESCE(qp.MIN_BOUNDARY, fy.MIN_BOUNDARY) AS MIN_BOUNDARY,
        COALESCE(qp.MEDIAN_BM,    fy.MEDIAN_BM)    AS MEDIAN_BM,
        COALESCE(qp.MAX_BOUNDARY, fy.MAX_BOUNDARY) AS MAX_BOUNDARY
    FROM full_year_percentiles fy
    LEFT JOIN quarter_percentiles qp
        ON fy.product_id = qp.product_id
        AND fy.warehouse_id = qp.warehouse_id
),

recent_data AS (
    SELECT *, EXP(-0.023 * days_ago) AS time_weight
    FROM iqr_cleaned
    WHERE days_ago <= 120
),

margin_bins AS (
    SELECT
        product_id, warehouse_id,
        ROUND(daily_margin, 2) AS margin_bin,
        SUM(nmv * time_weight)          AS weighted_nmv,
        SUM(gross_profit * time_weight) AS weighted_gp,
        COUNT(*)                        AS obs_count
    FROM recent_data
    GROUP BY product_id, warehouse_id, ROUND(daily_margin, 2)
),

smoothed_performance AS (
    SELECT
        product_id, warehouse_id, margin_bin, obs_count,
        AVG(weighted_nmv) OVER (
            PARTITION BY product_id, warehouse_id ORDER BY margin_bin
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS smooth_nmv,
        AVG(weighted_gp) OVER (
            PARTITION BY product_id, warehouse_id ORDER BY margin_bin
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS smooth_gp,
        SUM(obs_count) OVER (
            PARTITION BY product_id, warehouse_id ORDER BY margin_bin
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS smooth_obs
    FROM margin_bins
),

optimal_margin AS (
    SELECT
        product_id, warehouse_id,
        margin_bin AS optimal_bm
    FROM smoothed_performance
    WHERE smooth_obs >= 2
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id, warehouse_id
        ORDER BY smooth_gp DESC, smooth_nmv DESC
    ) = 1
)

SELECT
    COALESCE(wp.product_id, om.product_id) AS product_id,
    COALESCE(wp.warehouse_id, om.warehouse_id) AS warehouse_id,
    om.optimal_bm,
    wp.MIN_BOUNDARY,
    wp.MAX_BOUNDARY,
    wp.MEDIAN_BM
FROM weighted_percentiles wp
FULL OUTER JOIN optimal_margin om
    ON wp.product_id = om.product_id
    AND wp.warehouse_id = om.warehouse_id
WHERE COALESCE(wp.MIN_BOUNDARY, om.optimal_bm) IS NOT NULL
'''


def get_margin_tiers() -> pd.DataFrame:
    """Compute margin boundaries and calculate margin tiers from Snowflake.
    Includes cascading fallback: warehouse -> region -> global."""
    print("\n" + "="*70)
    print("FETCHING MARGIN TIERS")
    print("="*70)
    print(f"Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time")

    print("\nStep 1: Computing margin boundaries from sales data...")
    df_margin_boundaries = query_snowflake(MARGIN_BOUNDARIES_QUERY)
    print(f"    Loaded {len(df_margin_boundaries)} records (per product per warehouse)")

    if len(df_margin_boundaries) == 0:
        print("    Warning: No margin boundaries found!")
        return pd.DataFrame()

    print("\nStep 2: Mapping warehouses to cohorts...")
    df = df_margin_boundaries.copy()
    df = df.drop_duplicates(subset=['product_id', 'warehouse_id'])
    print(f"    Records after dedup: {len(df)}")

    print("\nStep 2a: Building scaffold of all active product-warehouse pairs...")
    df_scaffold = query_snowflake(f'''
        SELECT DISTINCT pso.product_id, COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id
        FROM product_sales_order pso
        LEFT JOIN (SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)) pw
            ON pw.child_id = pso.warehouse_id
        JOIN sales_orders so ON so.id = pso.sales_order_id
        WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 365
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
    ''')
    before = len(df)
    df = df_scaffold.merge(df, on=['product_id', 'warehouse_id'], how='left')
    added = len(df) - before
    print(f"    Scaffold: {len(df_scaffold)} active pairs, added {added} rows without warehouse-level boundaries")

    WH_REGION = {
        1: 'Cairo', 236: 'Giza', 962: 'Giza', 337: 'Delta West',
        8: 'Delta West', 339: 'Delta East', 170: 'Delta East',
        501: 'Upper Egypt', 401: 'Upper Egypt', 703: 'Upper Egypt',
        632: 'Upper Egypt', 797: 'Alexandria', 343: 'Giza', 467: 'Cairo',
    }
    boundary_cols = ['min_boundary', 'max_boundary', 'median_bm', 'optimal_bm']

    def _needs_fallback(frame):
        both_negative = (frame['min_boundary'] < 0) & (frame['max_boundary'] < 0)
        any_missing = frame['min_boundary'].isna() | frame['max_boundary'].isna()
        return both_negative | any_missing

    df['boundary_source'] = 'warehouse'
    bad = _needs_fallback(df)
    bad_count = bad.sum()
    print(f"\nStep 2b: Cascading fallback for bad boundaries...")
    print(f"    {bad_count} product-warehouse rows need fallback (both < 0 or missing)")

    if bad_count > 0:
        df['_region'] = df['warehouse_id'].map(WH_REGION)
        df_region = get_margin_boundaries_region()
        df = df.merge(df_region, left_on=['product_id', '_region'], right_on=['product_id', 'region'],
                      how='left', suffixes=('', '_rgn'))
        for col in boundary_cols:
            rgn_col = col + '_rgn'
            if rgn_col in df.columns:
                df.loc[bad, col] = df.loc[bad, rgn_col]
        df.loc[bad, 'boundary_source'] = 'region'
        drop_cols = [c + '_rgn' for c in boundary_cols]
        if 'region_rgn' in df.columns: drop_cols.append('region_rgn')
        if 'region' in df.columns and 'region' not in ['product_id', 'warehouse_id']:
            drop_cols.append('region')
        drop_cols.append('_region')
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors='ignore')

        bad = _needs_fallback(df)
        still_bad = bad.sum()
        print(f"    After region fallback: {still_bad} still bad")

        if still_bad > 0:
            df_global = get_margin_boundaries_global()
            df = df.merge(df_global, on='product_id', how='left', suffixes=('', '_glb'))
            for col in boundary_cols:
                glb_col = col + '_glb'
                if glb_col in df.columns:
                    df.loc[bad, col] = df.loc[bad, glb_col]
            df.loc[bad, 'boundary_source'] = 'global'
            df.drop(columns=[c + '_glb' for c in boundary_cols if c + '_glb' in df.columns],
                    inplace=True, errors='ignore')

            final_bad = _needs_fallback(df).sum()
            print(f"    After global fallback: {final_bad} still bad")

    region_count = (df['boundary_source'] == 'region').sum()
    global_count = (df['boundary_source'] == 'global').sum()
    if region_count > 0 or global_count > 0:
        print(f"    Fallback summary: {region_count} region, {global_count} global")

    print("\nStep 3: Calculating margin tiers...")
    df['effective_min_margin'] = df[['min_boundary', 'optimal_bm']].min(axis=1)
    df['margin_step'] = (df['max_boundary'] - df['effective_min_margin']) / 4

    df['margin_tier_below'] = df['effective_min_margin']
    df['margin_tier_1'] = df['effective_min_margin'] + (0.5 * df['margin_step'])
    df['margin_tier_2'] = df['effective_min_margin'] + df['margin_step']
    df['margin_tier_3'] = df['effective_min_margin'] + 2 * df['margin_step']
    df['margin_tier_4'] = df['effective_min_margin'] + 3 * df['margin_step']
    df['margin_tier_5'] = df['max_boundary']
    df['margin_tier_above_1'] = df['max_boundary'] + df['margin_step']
    df['margin_tier_above_2'] = df['max_boundary'] + 2 * df['margin_step']

    output_cols = [
        'product_id', 'warehouse_id',
        'optimal_bm', 'min_boundary', 'max_boundary', 'median_bm',
        'effective_min_margin', 'margin_step',
        'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
        'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
        'boundary_source'
    ]
    df = df[[c for c in output_cols if c in df.columns]]

    print("\n" + "="*70)
    print("MARGIN TIERS COMPLETE")
    print("="*70)
    print(f"Total records: {len(df)}")
    return df


MARKET_SIGNAL_QUERY = '''
WITH
daily_prices AS (
    SELECT
        product_id, warehouse_id, cohort_id, region,
        created_at AS price_date, wac_p, percentile_50, market_50
    FROM MATERIALIZED_VIEWS.Pricing_data_extraction
    WHERE created_at >= CURRENT_DATE - 60
        AND percentile_50 IS NOT NULL
),

with_changes AS (
    SELECT *,
        percentile_50 - LAG(percentile_50) OVER (
            PARTITION BY product_id, warehouse_id ORDER BY price_date
        ) AS daily_change
    FROM daily_prices
),

with_technicals AS (
    SELECT
        product_id, warehouse_id, cohort_id, region,
        price_date, wac_p, percentile_50, market_50, daily_change,
        AVG(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS sma_7,
        AVG(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS sma_14,
        AVG(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS sma_30,
        FIRST_VALUE(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS price_start_7,
        FIRST_VALUE(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS price_start_14,
        FIRST_VALUE(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS price_start_30,
        MIN(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS low_30d,
        MAX(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS high_30d,
        STDDEV(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW
        ) / NULLIF(AVG(percentile_50) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW), 0) AS cv_14d,
        COUNT(*) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS data_points_30d,
        SUM(CASE WHEN daily_change > 0 THEN 1 ELSE 0 END) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS up_days_14,
        SUM(CASE WHEN daily_change < 0 THEN 1 ELSE 0 END) OVER (PARTITION BY product_id, warehouse_id ORDER BY price_date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS down_days_14
    FROM with_changes
),

latest AS (
    SELECT *
    FROM with_technicals
    QUALIFY price_date = MAX(price_date) OVER (PARTITION BY product_id, warehouse_id)
)

SELECT
    product_id, warehouse_id, cohort_id, region,
    price_date AS last_observed, data_points_30d,
    ROUND(wac_p, 2) AS wac,
    ROUND(percentile_50, 2) AS market_price,
    ROUND((percentile_50 - wac_p) / NULLIF(percentile_50, 0) * 100, 1) AS market_margin_pct,
    ROUND(sma_7, 2) AS sma_7d, ROUND(sma_14, 2) AS sma_14d, ROUND(sma_30, 2) AS sma_30d,
    ROUND((percentile_50 - price_start_7) / NULLIF(price_start_7, 0) * 100, 2) AS pct_change_7d,
    ROUND((percentile_50 - price_start_14) / NULLIF(price_start_14, 0) * 100, 2) AS pct_change_14d,
    ROUND((percentile_50 - price_start_30) / NULLIF(price_start_30, 0) * 100, 2) AS pct_change_30d,
    CASE
        WHEN sma_7 > sma_14 AND sma_14 > sma_30 AND percentile_50 > sma_7 THEN 'STRONG UPTREND'
        WHEN sma_7 > sma_30 AND percentile_50 > sma_14 THEN 'UPTREND'
        WHEN sma_7 > sma_30 THEN 'WEAK UPTREND'
        WHEN sma_7 < sma_14 AND sma_14 < sma_30 AND percentile_50 < sma_7 THEN 'STRONG DOWNTREND'
        WHEN sma_7 < sma_30 AND percentile_50 < sma_14 THEN 'DOWNTREND'
        WHEN sma_7 < sma_30 THEN 'WEAK DOWNTREND'
        WHEN ABS(sma_7 - sma_30) / NULLIF(sma_30, 0) < 0.005 THEN 'SIDEWAYS'
        ELSE 'NEUTRAL'
    END AS trend_signal,
    CASE
        WHEN percentile_50 > sma_7 AND sma_7 > sma_14 THEN 'ACCELERATING UP'
        WHEN percentile_50 < sma_7 AND sma_7 > sma_14 THEN 'DECELERATING (was rising)'
        WHEN percentile_50 < sma_7 AND sma_7 < sma_14 THEN 'ACCELERATING DOWN'
        WHEN percentile_50 > sma_7 AND sma_7 < sma_14 THEN 'DECELERATING (was falling)'
        ELSE 'FLAT'
    END AS momentum,
    ROUND((percentile_50 - low_30d) / NULLIF(high_30d - low_30d, 0) * 100, 1) AS range_position,
    ROUND(low_30d, 2) AS support_30d,
    ROUND(high_30d, 2) AS resistance_30d,
    ROUND(cv_14d * 100, 2) AS volatility_pct,
    up_days_14, down_days_14
FROM latest
WHERE data_points_30d >= 5
'''


def get_market_signals() -> pd.DataFrame:
    """Get market trend signals for all active SKUs.
    Queries 60 days of Pricing_data_extraction history and computes technical indicators."""
    print("\n" + "="*70)
    print("FETCHING MARKET SIGNALS (Trend Analysis)")
    print("="*70)
    print(f"Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time")
    print("Analyzing 60-day price history from Pricing_data_extraction...")

    df = query_snowflake(MARKET_SIGNAL_QUERY)

    if len(df) == 0:
        print("  WARNING: No market signals found!")
        return pd.DataFrame()

    print(f"\n  Total SKUs analyzed: {len(df)}")

    if 'trend_signal' in df.columns:
        print(f"\n  --- Trend Distribution ---")
        for signal, count in df['trend_signal'].value_counts().items():
            print(f"    {signal:25s}: {count:4d} SKUs")

    if 'pct_change_30d' in df.columns:
        avg_chg = df['pct_change_30d'].mean()
        big_up = len(df[df['pct_change_30d'] > 5])
        big_down = len(df[df['pct_change_30d'] < -5])
        print(f"\n  --- Market Overview ---")
        print(f"    Avg 30d price change:   {avg_chg:+.2f}%")
        print(f"    SKUs up >5% (30d):      {big_up}")
        print(f"    SKUs down >5% (30d):    {big_down}")

    print("\n" + "="*70)
    print("MARKET SIGNALS COMPLETE")
    print("="*70)
    return df


print('Margin tiers + market signals defined')

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def apply_regional_fallback(df_prices):
    """For each (product, target_region) missing from df_prices,
    borrow prices from the first available fallback region."""
    all_products = df_prices['product_id'].unique()
    fallback_rows = []

    for pid in all_products:
        pid_prices = df_prices[df_prices['product_id'] == pid]
        pid_regions = set(pid_prices['region'].unique())

        for target_region in ALL_REGIONS:
            if target_region in pid_regions:
                continue
            for fb_region in REGIONAL_FALLBACK.get(target_region, []):
                fb_data = pid_prices[pid_prices['region'] == fb_region]
                if not fb_data.empty:
                    rows = fb_data.copy()
                    rows['region'] = target_region
                    fallback_rows.append(rows)
                    break

    if fallback_rows:
        return pd.concat([df_prices] + fallback_rows, ignore_index=True)
    return df_prices


def subdivide_price_list(prices, wac, target_margin):
    """Insert intermediate prices when the margin gap between adjacent tiers
    exceeds MAX_MARGIN_GAP_PCT * target_margin."""
    if wac <= 0 or target_margin <= 0 or len(prices) < 2:
        return prices

    max_gap = MAX_MARGIN_GAP_PCT * target_margin
    result = [prices[0]]

    for i in range(1, len(prices)):
        p_lo, p_hi = result[-1], prices[i]
        if p_hi <= wac:
            result.append(p_hi)
            continue

        # If lower price is at or below WAC, start subdivision from margin=0
        m_lo = max((p_lo - wac) / p_lo, 0) if p_lo > wac else 0
        m_hi = (p_hi - wac) / p_hi
        gap = m_hi - m_lo

        if gap > max_gap:
            n_steps = int(np.ceil(gap / max_gap))
            margin_step = gap / n_steps
            for s in range(1, n_steps):
                mid_margin = m_lo + s * margin_step
                if 0 < mid_margin < 1:
                    mid_price = np.round(wac / (1 - mid_margin) * 4) / 4
                    result.append(mid_price)

        result.append(p_hi)

    return sorted(set(result))


def expand_single_price_margin_steps(price, wac, target_margin):
    """Generate tiers by stepping in margin increments centered on the single price.
    Step size = MAX_MARGIN_GAP_PCT * target_margin. 2 steps below + anchor + 2 steps above."""
    if wac <= 0 or price <= wac:
        return [price]
    
    margin_at_price = (price - wac) / price
    step = target_margin * MAX_MARGIN_GAP_PCT if target_margin > 0 else 0.03
    
    tiers = set()
    for i in [-2, -1, 0, 1, 2]:
        m = margin_at_price + i * step
        if 0 < m < 0.99:
            tiers.add(np.round(wac / (1 - m) * 4) / 4)
    tiers.add(price)  # always include anchor
    
    return sorted(tiers)


def compute_brand_price_tiers(df_all, df_wac, df_products, df_targets):
    """Compute brand-level price tiers for SKUs without direct market data.
    Groups all raw prices by (brand, cat, region), computes margin percentiles,
    and converts to price tiers using each SKU's own WAC."""
    
    existing_keys = set(zip(df_all['product_id'], df_all['region']))
    
    # df_all already has wac_p from the pipeline merge
    brand_prices = df_all[['product_id', 'region', 'price', 'wac_p']].copy()
    brand_prices = brand_prices.merge(df_products[['product_id', 'brand', 'cat']], on='product_id', how='left')
    brand_prices = brand_prices[brand_prices['wac_p'] > 0]
    brand_prices['margin'] = (brand_prices['price'] - brand_prices['wac_p']) / brand_prices['price']
    brand_prices = brand_prices[brand_prices['margin'] > 0]
    
    if len(brand_prices) == 0:
        return pd.DataFrame()
    
    brand_percs = brand_prices.groupby(['brand', 'cat', 'region']).agg(
        perc_15=('margin', lambda x: np.percentile(x, 15)),
        perc_25=('margin', lambda x: np.percentile(x, 25)),
        perc_50=('margin', lambda x: np.percentile(x, 50)),
        perc_75=('margin', lambda x: np.percentile(x, 75)),
        perc_95=('margin', lambda x: np.percentile(x, 95)),
        n_prices=('margin', 'count'),
    ).reset_index()
    brand_percs = brand_percs[brand_percs['n_prices'] >= 3]
    
    if len(brand_percs) == 0:
        return pd.DataFrame()
    
    all_products = df_products.merge(df_wac, on='product_id', how='inner')
    all_products = all_products.merge(df_targets, on=['brand', 'cat'], how='left')
    all_products['target_margin'] = all_products['target_bm'].fillna(
        all_products['cat_target_margin']).fillna(DEFAULT_TARGET_MARGIN)
    
    brand_rows = []
    for _, prod in all_products.iterrows():
        wac = prod['wac_p']
        if wac <= 0:
            continue
        for region in ALL_REGIONS:
            if (prod['product_id'], region) in existing_keys:
                continue
            bp = brand_percs[(brand_percs['brand'] == prod['brand']) & 
                            (brand_percs['cat'] == prod['cat']) &
                            (brand_percs['region'] == region)]
            if bp.empty:
                continue
            bp = bp.iloc[0]
            prices = []
            for perc_col in ['perc_15', 'perc_25', 'perc_50', 'perc_75', 'perc_95']:
                m = bp[perc_col]
                if 0 < m < 1:
                    p = np.round(wac / (1 - m) * 4) / 4
                    if p >= 0.9 * wac:
                        prices.append(p)
            if prices:
                for p in sorted(set(prices)):
                    brand_rows.append({
                        'product_id': prod['product_id'],
                        'region': region,
                        'price': p,
                        'source': 'brand_fallback',
                        'wac_p': wac,
                        'target_margin': prod['target_margin'],
                    })
    
    return pd.DataFrame(brand_rows) if brand_rows else pd.DataFrame()


def expand_to_cohorts(df_market):
    """Expand region rows to cohort_id rows for module merging."""
    rows = []
    for _, row in df_market.iterrows():
        for cid in REGION_COHORT_MAP.get(row['region'], []):
            r = row.to_dict()
            r['cohort_id'] = cid
            rows.append(r)
    return pd.DataFrame(rows)


def tiers_to_percentiles(df_v2):
    """Derive legacy percentile columns from V2 price_tiers.
    Returns DataFrame with product_id, region, and market columns."""
    rows = []
    for _, row in df_v2.iterrows():
        tiers = row['price_tiers']
        if not tiers:
            continue
        wac = row['wac_p']
        p_min = min(tiers)
        p_25 = np.percentile(tiers, 25)
        p_50 = np.percentile(tiers, 50)
        p_75 = np.percentile(tiers, 75)
        p_max = max(tiers)
        rows.append({
            'product_id': row['product_id'],
            'region': row['region'],
            'market_min': p_min,
            'market_25': p_25,
            'market_50': p_50,
            'market_75': p_75,
            'market_max': p_max,
            'minimum': p_min,
            'percentile_25': p_25,
            'percentile_50': p_50,
            'percentile_75': p_75,
            'maximum': p_max,
            'below_market': (p_min - wac) / p_min if p_min > 0 else None,
            'above_market': (p_max - wac) / p_max if p_max > 0 else None,
            'market_data_source': row.get('market_data_source', 'sku'),
        })
    return pd.DataFrame(rows)


def get_market_data_legacy():
    """Legacy interface: derives percentile columns from V2 price_tiers.
    Brand fallback is already included in V2 output.
    Returns cohort-level data (expanded from region) for backward compatibility."""
    df_v2 = get_market_data_v2()
    df_legacy = tiers_to_percentiles(df_v2)
    df_legacy = expand_to_cohorts(df_legacy)
    print(f"Legacy output: {len(df_legacy)} rows from V2 price_tiers")
    return df_legacy

get_market_data = get_market_data_legacy


print('Helper functions defined')

In [ ]:
# =============================================================================
# MAIN PIPELINE
# =============================================================================
def get_market_data_v2():
    """Fetch and process market prices from all sources.
    Returns one row per (product_id, region) with sorted price_tiers list."""

    print('=' * 60)
    print('MARKET DATA V2')
    print('=' * 60)

    # ---- 1. Fetch raw data ----
    print('\n1. Fetching raw data...')

    print('  1a. Ben Soliman (savvy)...')
    df_ben = query_snowflake(V2_BEN_QUERY)
    print(f'      {len(df_ben)} products')

    print('  1a2. Ben Soliman (in-house mapping)...')
    df_ben_inhouse = query_snowflake(V2_BEN_INHOUSE_QUERY)
    print(f'      {len(df_ben_inhouse)} products')

    print('  1b. Marketplace...')
    df_mp = query_snowflake(V2_MARKETPLACE_QUERY)
    print(f'      {len(df_mp)} rows')

    print('  1c. Scraped...')
    df_sc = query_snowflake(V2_SCRAPED_QUERY)
    print(f'      {len(df_sc)} rows')

    print('  1d. WAC...')
    df_wac = query_snowflake(V2_WAC_QUERY)
    print(f'      {len(df_wac)} products')

    print('  1e. Target margins...')
    df_targets = query_snowflake(V2_TARGET_MARGINS_QUERY)
    print(f'      {len(df_targets)} brand-cat targets')

    print('  1f. Product info...')
    df_products = query_snowflake(V2_PRODUCT_QUERY)
    print(f'      {len(df_products)} products')

    print('  1g. Commercial groups...')
    df_groups = query_snowflake(V2_GROUPS_QUERY)
    print(f'      {len(df_groups)} group assignments')

    print('  1h. ATH margins...')
    df_ath = query_snowflake(V2_ATH_MARGIN_QUERY)
    print(f'      {len(df_ath)} products with ATH margin')

    # ---- 2. Expand Ben Soliman to all regions ----
    print('\n2. Expanding Ben Soliman to all regions...')
    ben_rows = []
    for _, row in df_ben.iterrows():
        for r in ALL_REGIONS:
            ben_rows.append({'product_id': row['product_id'], 'region': r,
                             'price': row['price'], 'source': 'ben_soliman'})
    for _, row in df_ben_inhouse.iterrows():
        for r in ALL_REGIONS:
            ben_rows.append({'product_id': row['product_id'], 'region': r,
                             'price': row['price'], 'source': 'ben_inhouse'})
    df_ben_expanded = pd.DataFrame(ben_rows)
    print(f'   {len(df_ben_expanded)} rows (savvy: {len(df_ben)*len(ALL_REGIONS)}, in-house: {len(df_ben_inhouse)*len(ALL_REGIONS)})')

    # ---- 3. Combine all sources ----
    print('\n3. Combining all sources...')
    cols = ['product_id', 'region', 'price', 'source']
    df_all = pd.concat([
        df_ben_expanded[cols],
        df_mp[cols],
        df_sc[cols],
    ], ignore_index=True)
    print(f'   {len(df_all)} total price points')

    # ---- 4. Regional fallback (marketplace + scraped only) ----
    print('\n4. Applying regional fallback...')
    ben_sources = {'ben_soliman', 'ben_inhouse'}
    df_regional = df_all[~df_all['source'].isin(ben_sources)].copy()
    df_ben_part = df_all[df_all['source'].isin(ben_sources)].copy()
    df_regional = apply_regional_fallback(df_regional)
    df_all = pd.concat([df_ben_part, df_regional], ignore_index=True)
    print(f'   {len(df_all)} total after fallback')

    # ---- 5. WAC floor filter ----
    print('\n5. WAC floor filter (>= 0.9 * WAC)...')
    df_all = df_all.merge(df_wac, on='product_id', how='inner')
    before = len(df_all)
    df_all = df_all[df_all['price'] >= 0.9 * df_all['wac_p']].copy()
    print(f'   {before} -> {len(df_all)} (removed {before - len(df_all)})')

    # ---- 6. Target margins (brand-cat -> cat -> default 10%) ----
    print('\n6. Target margins...')
    df_all = df_all.merge(df_products[['product_id', 'brand', 'cat']], on='product_id', how='left')
    df_all = df_all.merge(df_targets, on=['brand', 'cat'], how='left')
    df_all['target_margin'] = (
        df_all['target_bm']
        .fillna(df_all['cat_target_margin'])
        .fillna(DEFAULT_TARGET_MARGIN)
    )
    df_all.drop(columns=['target_bm', 'cat_target_margin'], inplace=True, errors='ignore')
    with_brand = df_all['target_margin'].notna().sum()
    print(f'   {with_brand} rows with resolved target margin')

    # ---- 7. Deduplicate ----
    print('\n7. Deduplicating...')
    df_all['price'] = df_all['price'].round(2)
    before = len(df_all)
    df_all = df_all.drop_duplicates(subset=['product_id', 'region', 'price'])
    print(f'   {before} -> {len(df_all)}')

    # ---- 8. Brand fallback for SKUs with no market data ----
    print('\n8. Brand fallback for SKUs without market data...')
    df_brand_fb = compute_brand_price_tiers(df_all, df_wac, df_products, df_targets)
    if len(df_brand_fb) > 0:
        df_brand_fb['brand'] = None
        df_brand_fb['cat'] = None
        df_brand_fb['group_id'] = None
        df_all = pd.concat([df_all, df_brand_fb[df_all.columns.intersection(df_brand_fb.columns)]], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=['product_id', 'region', 'price'])
        print(f'   Added {len(df_brand_fb)} brand fallback prices for {df_brand_fb["product_id"].nunique()} products')
    else:
        print('   No brand fallback needed')

    # ---- 9. Commercial groups ----
    print('\n9. Commercial group price union...')
    g_cols = df_groups.columns.tolist()
    group_col = 'group_id' if 'group_id' in g_cols else g_cols[0]
    product_col = 'product_id' if 'product_id' in g_cols else g_cols[1]
    df_gc = df_groups[[group_col, product_col]].rename(
        columns={group_col: 'group_id', product_col: 'product_id'}
    ).dropna(subset=['group_id'])

    df_all = df_all.merge(df_gc, on='product_id', how='left')
    has_group = df_all['group_id'].notna()

    if has_group.any():
        df_grouped = df_all[has_group].copy()
        df_ungrouped = df_all[~has_group].copy()

        group_prices = (
            df_grouped.groupby(['group_id', 'region'])['price']
            .apply(list).reset_index()
        )
        group_prices.columns = ['group_id', 'region', 'gp_list']

        group_products = df_gc.drop_duplicates()
        gx = group_products.merge(group_prices, on='group_id', how='inner')
        gx = gx.explode('gp_list').rename(columns={'gp_list': 'price'})
        gx['price'] = gx['price'].astype(float)
        gx = gx.merge(df_wac, on='product_id', how='inner')
        gx = gx[gx['price'] >= 0.9 * gx['wac_p']]
        gx = gx.merge(df_products[['product_id', 'brand', 'cat']], on='product_id', how='left')
        gx = gx.merge(df_targets, on=['brand', 'cat'], how='left')
        gx['target_margin'] = gx['target_bm'].fillna(gx['cat_target_margin']).fillna(DEFAULT_TARGET_MARGIN)
        gx['source'] = 'group_union'
        gx = gx[['product_id', 'region', 'price', 'source', 'wac_p', 'brand', 'cat',
                  'target_margin', 'group_id']]

        df_all = pd.concat([df_ungrouped, df_grouped, gx], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=['product_id', 'region', 'price'])
        print(f'   Expanded -> {len(df_all)} total after group union + dedup')
    else:
        print('   No commercial groups found')

    # ---- 10. Source counts + market_data_source ----
    source_counts = (
        df_all.groupby(['product_id', 'region'])['source']
        .nunique().reset_index()
        .rename(columns={'source': 'num_sources'})
    )
    # Track whether SKU has direct market data or brand fallback
    source_type = df_all.groupby(['product_id', 'region'])['source'].apply(
        lambda x: 'brand' if set(x) == {'brand_fallback'} else 'sku'
    ).reset_index().rename(columns={'source': 'market_data_source'})

    # ---- 11. Aggregate into price lists ----
    print('\n10. Building price tiers...')
    agg = df_all.groupby(['product_id', 'region']).agg(
        prices=('price', lambda x: sorted(set(x))),
        wac_p=('wac_p', 'first'),
        target_margin=('target_margin', 'first'),
    ).reset_index()

    agg = agg.merge(source_counts, on=['product_id', 'region'], how='left')
    agg['num_sources'] = agg['num_sources'].fillna(1).astype(int)
    agg = agg.merge(source_type, on=['product_id', 'region'], how='left')
    agg['market_data_source'] = agg['market_data_source'].fillna('sku')

    # ---- 10b. Expand single-price SKUs (two-stage) ----
    single_mask = agg['prices'].apply(len) == 1
    single_count = single_mask.sum()
    fb_expanded = 0
    margin_expanded = 0
    
    if single_count > 0:
        for idx in agg[single_mask].index:
            pid = agg.loc[idx, 'product_id']
            region = agg.loc[idx, 'region']
            single_price = agg.loc[idx, 'prices'][0]
            wac = agg.loc[idx, 'wac_p']
            tm = agg.loc[idx, 'target_margin']
            
            # Stage 1: Check fallback regions for additional prices
            expanded = False
            for fb_region in REGIONAL_FALLBACK.get(region, []):
                fb_row = agg[(agg['product_id'] == pid) & (agg['region'] == fb_region)]
                if fb_row.empty:
                    continue
                fb_prices = fb_row.iloc[0]['prices']
                if len(fb_prices) > 1 or (len(fb_prices) == 1 and fb_prices[0] != single_price):
                    agg.at[idx, 'prices'] = sorted(set([single_price] + fb_prices))
                    expanded = True
                    fb_expanded += 1
                    break
            
            # Stage 2: Margin-step expansion centered on the single price
            if not expanded:
                agg.at[idx, 'prices'] = expand_single_price_margin_steps(single_price, wac, tm)
                margin_expanded += 1
        
        print(f'   {single_count} single-price SKUs: {fb_expanded} expanded from fallback regions, {margin_expanded} expanded with margin steps')

    # ---- 10b2. Target margin + ATH margin price injection ----
    # Only for SKUs with actual market data — brand-fallback-only SKUs are left untouched
    agg = agg.merge(df_ath, on='product_id', how='left')
    print('\n   Injecting target margin + ATH margin anchor prices (market-data SKUs only)...')
    tm_injected = 0
    ath_injected = 0
    for idx in agg.index:
        if agg.loc[idx, 'market_data_source'] != 'sku':
            continue
        wac = agg.loc[idx, 'wac_p']
        tm = agg.loc[idx, 'target_margin']
        ath_m = agg.loc[idx, 'ath_margin'] if pd.notna(agg.loc[idx, 'ath_margin']) else None
        current = agg.loc[idx, 'prices']
        new_prices = set(current)

        if wac > 0 and 0 < tm < 1:
            tp = round(wac / (1 - tm) * 4) / 4
            if tp >= 0.9 * wac and tp not in new_prices:
                new_prices.add(tp)
                tm_injected += 1

        if ath_m is not None and wac > 0 and 0 < ath_m < 1:
            ath_price = round(wac / (1 - ath_m) * 4) / 4
            if ath_price >= 0.9 * wac and ath_price not in new_prices:
                new_prices.add(ath_price)
                ath_injected += 1

        if len(new_prices) > len(current):
            agg.at[idx, 'prices'] = sorted(new_prices)

    print(f'   Target margin: injected into {tm_injected} product-region combinations')
    print(f'   ATH margin: injected into {ath_injected} product-region combinations')
    agg.drop(columns=['ath_margin'], inplace=True)

    # ---- 10c. Step subdivision ----
    agg['price_tiers'] = agg.apply(
        lambda row: subdivide_price_list(row['prices'], row['wac_p'], row['target_margin']),
        axis=1
    )
    agg.drop(columns=['prices'], inplace=True)

    print(f'   {len(agg)} product x region combinations')
    print(f'   Avg tiers: {agg["price_tiers"].apply(len).mean():.1f}')
    print(f'   Median tiers: {agg["price_tiers"].apply(len).median():.0f}')

    # ---- 10d. Commercial price-up induced prices ----
    print('\n11. Commercial price-up induced prices...')
    df_price_ups = get_commercial_price_ups()
    if len(df_price_ups) > 0:
        df_price_ups['ratio'] = df_price_ups['wac_p'] / df_price_ups['wac1']
        df_price_ups['new_wac_p'] = df_price_ups['new_pp'] * df_price_ups['ratio']
        induced_count = 0
        for _, pu in df_price_ups.iterrows():
            pid = pu['product_id']
            new_wac_p = pu['new_wac_p']
            if pd.isna(new_wac_p) or new_wac_p <= 0:
                continue
            mask = agg['product_id'] == pid
            if not mask.any():
                continue
            for idx in agg[mask].index:
                tm = agg.loc[idx, 'target_margin']
                if pd.isna(tm) or tm >= 1 or tm <= 0:
                    tm = 0.10
                induced = round(new_wac_p / (1 - tm) * 4) / 4
                tiers = agg.loc[idx, 'price_tiers']
                if induced > 0 and induced not in tiers:
                    agg.at[idx, 'price_tiers'] = sorted(set(tiers + [induced]))
                    induced_count += 1
        print(f'   Added induced prices to {induced_count} product-region combinations from {len(df_price_ups)} price-ups')
    else:
        print('   No commercial price-ups found')

    # ---- 10e. Final rounding + dedup ----
    agg['price_tiers'] = agg['price_tiers'].apply(
        lambda tiers: sorted(set(round(p * 4) / 4 for p in tiers))
    )

    print('\n' + '=' * 60)
    print('MARKET DATA V2 COMPLETE')
    print('=' * 60)

    return agg[['product_id', 'region', 'price_tiers', 'wac_p', 'target_margin', 'num_sources', 'market_data_source']]


print('get_market_data_v2() defined')